# 09 — Model Explainability Lab
## bd_replica_crm · por qué predice, dónde funciona y cómo explicarlo

Objetivo:

```text
predicción → explicación global → explicación por segmento
→ explicación local → sensibilidad → insight accionable
```

Incluye coeficientes/odds ratios, permutation importance, partial dependence,
sensibilidad, explicación local por lead, error por proyecto/asesor/canal/medio,
estabilidad temporal y smart insights.

No demuestra causalidad; para eso están los notebooks causales/experimentales.


In [ ]:
from __future__ import annotations
import sys
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

cwd=Path.cwd().resolve()
PROJECT_ROOT=cwd if (cwd/"pyproject.toml").exists() else cwd.parent
if not (PROJECT_ROOT/"pyproject.toml").exists():
    raise RuntimeError("Ejecuta este notebook dentro de bd_replica_crm.")

SRC=PROJECT_ROOT/"src"
if str(SRC) not in sys.path:
    sys.path.insert(0,str(SRC))

from replica_cygnus.settings import load_settings
from replica_cygnus.connections import connect_postgres

settings=load_settings(PROJECT_ROOT)
conn=connect_postgres(settings)

pd.set_option("display.max_columns",180)
pd.set_option("display.max_rows",180)
pd.set_option("display.width",260)

def sql_df(sql,params=None):
    with conn.cursor() as cur:
        cur.execute(sql,params or ())
        cols=[d.name for d in cur.description]
        rows=cur.fetchall()
    return pd.DataFrame(rows,columns=cols)

print("DB:",settings.postgres.database)
print("Inicio:",datetime.now().astimezone().isoformat(timespec="seconds"))


## 1. Configuración


In [ ]:
TARGET="minuta_60d"
MODEL="LogisticRegression"
FEATURE_SET="commercial"
SEGMENT_FILTER=None
TEST_FRACTION=.20
RANDOM_STATE=42
TOP_N=20


## 2. Feature sets


In [ ]:
FEATURE_SETS={
"compact":[
    "hour_of_day","day_of_week","is_weekend",
    "client_prior_assignments_90d","days_since_previous_assignment"
],
"historical_rates":[
    "project_sep_rate_90d","project_minuta_rate_180d",
    "advisor_sep_rate_90d","advisor_minuta_rate_180d",
    "global_sep_rate_90d","global_minuta_rate_180d"
],
"commercial":[
    "codigo_proyecto","asesor","canal","medio",
    "hour_of_day","day_of_week","is_weekend",
    "client_prior_assignments_90d","days_since_previous_assignment",
    "project_leads_90d","project_sep_rate_90d","project_minuta_rate_180d",
    "advisor_leads_90d","advisor_sep_rate_90d","advisor_minuta_rate_180d",
    "global_sep_rate_90d","global_minuta_rate_180d"
]
}
features=FEATURE_SETS[FEATURE_SET]
features


## 3. Dataset


In [ ]:
cols=[
"evidence_key","lead_id","decision_at",
"codigo_proyecto","asesor","canal","medio",
"hour_of_day","day_of_week","is_weekend",
"client_prior_assignments_90d","days_since_previous_assignment",
"project_leads_90d","project_sep_rate_90d","project_minuta_rate_180d",
"advisor_leads_90d","advisor_sep_rate_90d","advisor_minuta_rate_180d",
"global_sep_rate_90d","global_minuta_rate_180d",
"separacion_14d","minuta_60d"
]

data=sql_df("SELECT "+",".join(cols)+" FROM features.lead_evidence")
data["decision_at"]=pd.to_datetime(data["decision_at"],utc=True,errors="coerce")

if SEGMENT_FILTER:
    for c,v in SEGMENT_FILTER.items():
        data=data[data[c].eq(v)]

data=data[data[TARGET].notna()].copy()
data=data.sort_values(["decision_at","evidence_key"]).reset_index(drop=True)

print("rows:",len(data))
print("target rate:",data[TARGET].mean())


## 4. Split temporal


In [ ]:
split_idx=int(len(data)*(1-TEST_FRACTION))
train=data.iloc[:split_idx].copy()
test=data.iloc[split_idx:].copy()

print("train:",len(train),"test:",len(test))
print("train max:",train["decision_at"].max())
print("test min:",test["decision_at"].min())


## 5. Pipeline y métricas


In [ ]:
categorical=[
    c for c in features
    if data[c].dtype=="object" or str(data[c].dtype).startswith("string")
]
numeric=[c for c in features if c not in categorical]

num_pipe=Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler())
])

cat_pipe=Pipeline([
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("ohe",OneHotEncoder(handle_unknown="ignore"))
])

prep=ColumnTransformer([
    ("num",num_pipe,numeric),
    ("cat",cat_pipe,categorical)
])

if MODEL=="LogisticRegression":
    estimator=LogisticRegression(max_iter=1500,random_state=RANDOM_STATE)
elif MODEL=="RandomForest":
    estimator=RandomForestClassifier(
        n_estimators=300,min_samples_leaf=10,n_jobs=-1,random_state=RANDOM_STATE
    )
else:
    estimator=GradientBoostingClassifier(random_state=RANDOM_STATE)

model=Pipeline([("prep",prep),("model",estimator)])

X_train=train[features]
y_train=train[TARGET].astype(int)
X_test=test[features]
y_test=test[TARGET].astype(int)

model.fit(X_train,y_train)
prob=model.predict_proba(X_test)[:,1]

metrics=pd.DataFrame([{
    "model":MODEL,
    "n_train":len(train),
    "n_test":len(test),
    "base_rate":y_test.mean(),
    "roc_auc":roc_auc_score(y_test,prob) if y_test.nunique()>1 else np.nan,
    "average_precision":average_precision_score(y_test,prob),
    "brier":brier_score_loss(y_test,prob)
}])
metrics


## 6. Permutation importance global


In [ ]:
perm=permutation_importance(
    model,X_test,y_test,
    scoring="average_precision",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

importance=pd.DataFrame({
    "feature":features,
    "importance_mean":perm.importances_mean,
    "importance_std":perm.importances_std
}).sort_values("importance_mean",ascending=False)

display(importance.head(TOP_N))

p=importance.head(TOP_N).sort_values("importance_mean")
plt.figure(figsize=(10,7))
plt.barh(p["feature"],p["importance_mean"])
plt.xlabel("Permutation importance")
plt.title("Importancia global")
plt.tight_layout()
plt.show()


## 7. Coeficientes y odds ratios


In [ ]:
coef_table=pd.DataFrame()

if MODEL=="LogisticRegression":
    names=model.named_steps["prep"].get_feature_names_out()
    coefs=model.named_steps["model"].coef_[0]

    coef_table=pd.DataFrame({
        "feature_encoded":names,
        "coef":coefs,
        "odds_ratio":np.exp(coefs)
    })
    coef_table["abs_coef"]=coef_table["coef"].abs()
    coef_table=coef_table.sort_values("abs_coef",ascending=False)

    display(coef_table.head(40))
else:
    print("Disponible cuando MODEL='LogisticRegression'.")


## 8. Partial dependence manual


In [ ]:
PDP_FEATURE="project_sep_rate_90d"

if PDP_FEATURE in numeric and len(X_test):
    lo=X_test[PDP_FEATURE].quantile(.05)
    hi=X_test[PDP_FEATURE].quantile(.95)
    values=np.linspace(lo,hi,20)

    rows=[]
    for v in values:
        tmp=X_test.copy()
        tmp[PDP_FEATURE]=v
        rows.append({
            "value":v,
            "avg_prediction":model.predict_proba(tmp)[:,1].mean()
        })

    pdp=pd.DataFrame(rows)
    display(pdp)

    plt.figure(figsize=(8,5))
    plt.plot(pdp["value"],pdp["avg_prediction"],marker="o")
    plt.xlabel(PDP_FEATURE)
    plt.ylabel("Predicción promedio")
    plt.title("Partial dependence")
    plt.show()
else:
    print("Selecciona una feature numérica válida.")


## 9. Sensibilidad de una predicción


In [ ]:
SENSITIVITY_FEATURE="project_sep_rate_90d"

if SENSITIVITY_FEATURE in numeric and len(test):
    row=test.iloc[[0]][features].copy()
    original=float(row[SENSITIVITY_FEATURE].iloc[0])
    std=float(train[SENSITIVITY_FEATURE].std(skipna=True))

    scenarios=[("-1sd",original-std),("base",original),("+1sd",original+std)]
    sens=[]

    for label,v in scenarios:
        tmp=row.copy()
        tmp[SENSITIVITY_FEATURE]=v
        sens.append({
            "scenario":label,
            "feature_value":v,
            "prediction":model.predict_proba(tmp)[:,1][0]
        })

    sensitivity=pd.DataFrame(sens)
    display(sensitivity)


## 10. Explicación local por lead


In [ ]:
LOCAL_INDEX=0

local_row=test.iloc[[LOCAL_INDEX]].copy()
local_X=local_row[features].copy()
base_prob=model.predict_proba(local_X)[:,1][0]

local_effects=[]

for c in numeric:
    if pd.isna(local_X[c].iloc[0]):
        continue

    replacement=train[c].median()
    tmp=local_X.copy()
    tmp[c]=replacement
    p=model.predict_proba(tmp)[:,1][0]

    local_effects.append({
        "feature":c,
        "actual":local_X[c].iloc[0],
        "reference":replacement,
        "prediction_actual":base_prob,
        "prediction_if_reference":p,
        "delta":base_prob-p
    })

local_explanation=pd.DataFrame(local_effects)
local_explanation["abs_delta"]=local_explanation["delta"].abs()
local_explanation=local_explanation.sort_values("abs_delta",ascending=False)

print("Lead:",local_row["lead_id"].iloc[0])
print("Prediction:",base_prob)
print("Observed:",local_row[TARGET].iloc[0])
display(local_explanation.head(TOP_N))


## 11. Explicación por proyecto


In [ ]:
pred=test[
    ["evidence_key","decision_at","codigo_proyecto","asesor","canal","medio",TARGET]
].copy()

pred["prob"]=prob
pred["error"]=pred["prob"]-pred[TARGET]

segment_project=(
    pred.groupby("codigo_proyecto",dropna=False,as_index=False)
    .agg(
        n=(TARGET,"size"),
        observed=(TARGET,"mean"),
        predicted=("prob","mean"),
        mae=("error",lambda s:np.mean(np.abs(s))),
        bias=("error","mean")
    )
)

segment_project["calibration_gap_pp"]=(
    segment_project["predicted"]-segment_project["observed"]
)*100

display(
    segment_project[segment_project["n"]>=20]
    .sort_values("mae",ascending=False)
    .head(TOP_N)
)


## 12. Explicación por asesor / canal / medio


In [ ]:
segment_tables={}

for dim in ["asesor","canal","medio"]:
    t=(
        pred.groupby(dim,dropna=False,as_index=False)
        .agg(
            n=(TARGET,"size"),
            observed=(TARGET,"mean"),
            predicted=("prob","mean"),
            mae=("error",lambda s:np.mean(np.abs(s))),
            bias=("error","mean")
        )
    )
    t=t[t["n"]>=20].copy()
    segment_tables[dim]=t

    print("\\n",dim.upper())
    display(t.sort_values("mae",ascending=False).head(TOP_N))


## 13. Errores extremos


In [ ]:
false_positive=(
    pred[pred[TARGET].eq(0)]
    .sort_values("prob",ascending=False)
    .head(TOP_N)
)

false_negative=(
    pred[pred[TARGET].eq(1)]
    .sort_values("prob",ascending=True)
    .head(TOP_N)
)

print("Falsos positivos extremos")
display(false_positive)

print("Falsos negativos extremos")
display(false_negative)


## 14. Estabilidad temporal


In [ ]:
temporal=pred.copy()
temporal["month"]=temporal["decision_at"].dt.to_period("M").astype(str)

stability=(
    temporal.groupby("month",as_index=False)
    .agg(
        n=(TARGET,"size"),
        observed=(TARGET,"mean"),
        predicted=("prob","mean"),
        mae=("error",lambda s:np.mean(np.abs(s))),
        bias=("error","mean")
    )
)

display(stability.tail(24))

plt.figure(figsize=(12,5))
plt.plot(stability["month"],stability["observed"],marker="o",label="observed")
plt.plot(stability["month"],stability["predicted"],marker="o",label="predicted")
plt.xticks(rotation=45,ha="right")
plt.ylabel("Rate")
plt.title("Observed vs predicted por mes")
plt.legend()
plt.tight_layout()
plt.show()


## 15. Smart model insights


In [ ]:
insights=[]

if len(importance):
    r=importance.iloc[0]
    insights.append(
        f"Mayor importancia por permutación: {r['feature']} ({r['importance_mean']:.4f})."
    )

valid=segment_project[segment_project["n"]>=20]

if len(valid):
    worst=valid.sort_values("mae",ascending=False).iloc[0]
    insights.append(
        f"Mayor error por proyecto: {worst['codigo_proyecto']} "
        f"(MAE={worst['mae']:.3f}, n={int(worst['n'])})."
    )

    biased=valid.assign(abs_bias=valid["bias"].abs()).sort_values(
        "abs_bias",ascending=False
    ).iloc[0]

    direction="sobrepredice" if biased["bias"]>0 else "subpredice"
    insights.append(
        f"Mayor sesgo: {biased['codigo_proyecto']}; el modelo {direction}."
    )

if len(stability):
    recent=stability.iloc[-1]
    insights.append(
        f"Último periodo: observado={recent['observed']:.1%}, "
        f"predicho={recent['predicted']:.1%}, MAE={recent['mae']:.3f}."
    )

print("=== MODEL INSIGHTS ===")
for i,x in enumerate(insights,1):
    print(f"{i}. {x}")


## 16. Gate de explicabilidad


In [ ]:
gates=[]

def gate(name,passed,detail):
    gates.append({
        "gate":name,
        "status":"PASS" if passed else "FAIL",
        "detail":detail
    })

auc=float(metrics.iloc[0]["roc_auc"])

gate("Modelo discrimina",pd.notna(auc) and auc>.50,f"AUC={auc:.3f}")
gate("Permutation importance",len(importance)>0,f"features={len(importance)}")
gate(
    "Error segmentado evaluable",
    len(segment_project[segment_project["n"]>=20])>0,
    f"segments={len(segment_project[segment_project['n']>=20])}"
)
gate(
    "Split temporal",
    train["decision_at"].max()<test["decision_at"].min(),
    "train anterior a test"
)

gate_table=pd.DataFrame(gates)
gate_table


## 17. Tabla ejecutiva


In [ ]:
executive_table=(
    segment_project[segment_project["n"]>=20][
        [
            "codigo_proyecto","n","observed","predicted",
            "calibration_gap_pp","mae","bias"
        ]
    ]
    .sort_values("mae",ascending=False)
)

executive_table.head(TOP_N)


## 18. Export opcional


In [ ]:
EXPORT=False

if EXPORT:
    out=PROJECT_ROOT/"reports"/"model_explainability"
    out.mkdir(parents=True,exist_ok=True)

    metrics.to_csv(out/"metrics.csv",index=False)
    importance.to_csv(out/"permutation_importance.csv",index=False)
    segment_project.to_csv(out/"project_explainability.csv",index=False)
    stability.to_csv(out/"temporal_stability.csv",index=False)
    local_explanation.to_csv(out/"local_explanation.csv",index=False)
    gate_table.to_csv(out/"explainability_gates.csv",index=False)

    if len(coef_table):
        coef_table.to_csv(out/"coefficients_odds_ratios.csv",index=False)

    print("Export:",out)
else:
    print("EXPORT=False")


## 19. Qué sí y qué no afirma este notebook

Sí:
- qué variables pesan más;
- dónde el modelo se equivoca;
- dónde sobre/subpredice;
- cómo cambia una predicción ante perturbaciones;
- qué explica una predicción concreta.

No:
- que cambiar una variable cause un mejor resultado.

Para causalidad usa `03_causal_inference_lab.ipynb`,
`05_experiment_uplift_lab.ipynb` o el notebook del piloto.


In [ ]:
conn.close(); print("Conexión cerrada.")
